## nn.Module — Regression

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler


In [5]:
## Data
X, y = fetch_california_housing(return_X_y=True)
print(X.shape)
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # (N,) → (N,1)

dataset= TensorDataset(X_t, y_t)

train_data, val_data, test_data = random_split(dataset, [13000, 4000, 3640])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=256)
test_loader  = DataLoader(test_data,  batch_size=256)

(20640, 8)


## Model

In [6]:
# Model
class HousingRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.out(x)

model     = HousingRegressor()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## Training loop

In [7]:
# Training loop
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                val_loss += criterion(model(X_batch), y_batch).item()
        print(f"Epoch {epoch+1:2d} | Val MSE: {val_loss/len(val_loader):.4f}")


Epoch 10 | Val MSE: 0.4385
Epoch 20 | Val MSE: 0.6487
Epoch 30 | Val MSE: 0.9311
Epoch 40 | Val MSE: 0.6606
Epoch 50 | Val MSE: 0.4979


## Test

In [8]:
# Test
model.eval()
test_loss = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        test_loss += criterion(model(X_batch), y_batch).item()
print(f"Test MSE: {test_loss/len(test_loader):.4f}")

Test MSE: 0.4891
